# Measurements API
Short reference for the measurement-related endpoints exposed by `GIDataClient`.

## 1) Setup

In [4]:
import os
from gi_data.dataclient import GIDataClient

BASE = os.getenv("GI_BASE", "http://127.0.0.1:8090")
USER = os.getenv("GI_USER", "admin")
PASS = os.getenv("GI_PASS", "admin")
client = GIDataClient(BASE, username=USER, password=PASS)

## 2) Get Measurements - all sources
`GET /history/structure/measurements`

In [5]:
all_meas = client.get_measurements()
print(f"Found {len(all_meas)} measurements")
for m in all_meas[:5]:
    print(m.id, m.name, m.source_id)

2026-07-27 16:48:24,119 - gi_data.infra.http - DEBUG - Request: GET http://127.0.0.1:8090/history/structure/measurements | Params: None | Payload: None


Found 4 measurements
28aecaae-3a16-425e-ab1e-1afec88404ac Testss f4379b36-232d-f546-5782-2582fba8e07d
2dd2ccd9-cd59-4d20-975c-114c11271bb6 2026-01-12_14-22-24 7c78b840-8bf7-005e-6cc4-3333dec86bf2
59f22a04-9823-484f-8dcd-213a7cfe9d99 2026-06-09_08-57-18 3acbeffa-00fd-11f1-9e6d-6c4b9078238e
82f873b2-b96c-483a-83d2-9117921c2192 2026-06-18_07-44-39 3acbeffa-00fd-11f1-9e6d-6c4b9078238e


## 3) Get Single Measurement
`GET /history/structure/measurements/<MeasurementID>`

In [3]:
mid = all_meas[0].id
meas = client.get_measurement(mid)
print(meas)

2026-07-27 15:17:22,240 - gi_data.infra.http - DEBUG - Request: GET http://127.0.0.1:8090/history/structure/measurements/28aecaae-3a16-425e-ab1e-1afec88404ac | Params: None | Payload: None


id='28aecaae-3a16-425e-ab1e-1afec88404ac' absolute_start=1764084243318.4 last_ts=1764084243659.6 sample_rate_hz=5000.0 source_id='f4379b36-232d-f546-5782-2582fba8e07d' name='Testss' available_time_sec=0.341200256 cfg_checksum='' data_storage='records' index=-1 is_removable=True kind='UDBF' max_time_sec=-1.0 start_date='2025-11-25T15:24:03Z' updated=False variables=None internal_id='7.2'


## 4) Get Source Measurements Advanced
`POST /history/structure/sources/<SourceID>/measurements`
Filter by time range, order, limit, measurement ids, or measurement metadata.

In [6]:
sources = client.list_history_sources()
sid = sources[0].id
meas_of_source = client.list_history_measurements(
    sid,
    order="DESC",
    limit=10,
    add_var_mapping=True,
    add_meas_metadata=False,
)
for m in meas_of_source:
    print(m.id, m.name)

2026-07-27 16:48:27,163 - gi_data.infra.http - DEBUG - Request: GET http://127.0.0.1:8090/history/structure/sources | Params: None | Payload: None
2026-07-27 16:48:27,169 - gi_data.infra.http - DEBUG - Request: POST http://127.0.0.1:8090/history/structure/sources/f4379b36-232d-f546-5782-2582fba8e07d/measurements | Params: None | Payload: {'Order': 'DESC', 'Limit': 10, 'AddVarMapping': True, 'AddMeasMetaData': False}


28aecaae-3a16-425e-ab1e-1afec88404ac Testss


## 5) Get Measurements Advanced - all sources
`POST /history/structure/measurements`

In [7]:
filtered = client.get_measurements_advanced(
    order="DESC",
    limit=20,
    meas_metadata_filter=[{"Key": "TestData", "Value": "test"}],
)
print(f"Filtered: {len(filtered)} measurements")

2026-07-27 16:48:30,931 - gi_data.infra.http - DEBUG - Request: POST http://127.0.0.1:8090/history/structure/measurements | Params: None | Payload: {'Order': 'DESC', 'AddVarMapping': True, 'AddMeasMetaData': False, 'Limit': 20, 'MeasMetaDataFilter': [{'Key': 'TestData', 'Value': 'test'}]}


Filtered: 0 measurements


## 6) Add Measurement Metadata
`POST /history/structure/measurements/<MeasurementID>/metadata`

In [9]:
client.add_measurement_metadata(
    mid,
    meas_name="MyNewMeasName",
    metadata=[
        {
            "Key": "TestData",
            "Value": "test",
            "Start": 1590393309991,
            "End": 1590393290000,
        }
    ],
)

2026-07-27 16:49:11,686 - gi_data.infra.http - DEBUG - Request: POST http://127.0.0.1:8090/history/structure/measurements/28aecaae-3a16-425e-ab1e-1afec88404ac/metadata | Params: None | Payload: {'MeasName': 'MyNewMeasName', 'MetaData': [{'Key': 'TestData', 'Value': 'test', 'Start': 1590393309991, 'End': 1590393290000}]}


## 7) Delete Measurement Metadata
`DELETE /history/structure/measurements/<MeasurementID>/metadata`

In [10]:
client.delete_measurement_metadata(mid)

2026-07-03 11:51:56,560 - gi_data.infra.http - DEBUG - Request: DELETE http://127.0.0.1:8090/history/structure/measurements/28aecaae-3a16-425e-ab1e-1afec88404ac/metadata | Params: None | Payload: None


## 8) Delete Measurement
`DELETE /history/structure/measurements/<MeasurementID>` - removes the measurement **and its data**. Use with care.

In [ ]:
# client.delete_measurement(mid)

## 9) Cleanup

In [ ]:
client.close()